In [ ]:
import pandas as pd

df = pd.read_csv('HAB_Final_Matched_Dataset_Labeled.csv')
df.head()

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATASET
# =========================================================

file_path = "/content/HAB_Final_Matched_Dataset_Labeled.csv"

df = pd.read_csv(file_path)

# =========================================================
# DISPLAY SETTINGS
# =========================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# =========================================================
# SELECT FEATURES
# =========================================================

features = [

    "B11",
    "B2",
    "B2_B3",
    "B3",
    "B3_B2",
    "B3_B4",
    "B4",
    "B5",
    "B5_B4",
    "B8",

    "CI",
    "FAI",
    "NDCI",
    "NDRE",
    "NDVI",
    "NDWI"
]

# =========================================================
# CHECK FEATURE AVAILABILITY
# =========================================================

missing_features = [

    col for col in features
    if col not in df.columns
]

if missing_features:

    print("\nMISSING FEATURE COLUMNS")
    print("=================================================")

    for col in missing_features:
        print(col)

else:

    print("\nALL SELECTED FEATURES ARE AVAILABLE")
    print("=================================================")

# =========================================================
# KEEP AVAILABLE FEATURES
# =========================================================

available_features = [

    col for col in features
    if col in df.columns
]

# =========================================================
# REMOVE MISSING VALUES
# =========================================================

df_stats = df[available_features].dropna()

print("\nDATA USED FOR STATISTICS")
print("=================================================")
print(df_stats.shape)

# =========================================================
# DESCRIPTIVE STATISTICS
# =========================================================

desc_table = (
    df_stats
    .describe()
    .T
)

# =========================================================
# ADD SKEWNESS
# =========================================================

desc_table["skewness"] = (
    df_stats
    .skew()
)

# =========================================================
# ADD MISSING VALUE COUNT
# =========================================================

desc_table["missing_values"] = (
    df[available_features]
    .isna()
    .sum()
)

# =========================================================
# SELECT IMPORTANT STATS
# =========================================================

desc_table = desc_table[
    [
        "count",
        "missing_values",
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
        "skewness"
    ]
]

# =========================================================
# RENAME COLUMNS
# =========================================================

desc_table = desc_table.rename(columns={

    "count": "Count",
    "missing_values": "Missing",
    "mean": "Mean",
    "std": "Std. Dev.",
    "min": "Min",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Max",
    "skewness": "Skewness"

})

# =========================================================
# ROUND VALUES
# =========================================================

desc_table = desc_table.round(4)

# =========================================================
# RESET INDEX
# =========================================================

desc_table = (
    desc_table
    .reset_index()
    .rename(columns={"index": "Feature"})
)

# =========================================================
# SORT FEATURES
# =========================================================

desc_table = desc_table.sort_values(
    by="Skewness",
    ascending=False
)

# =========================================================
# DISPLAY TABLE
# =========================================================

print("\nDESCRIPTIVE STATISTICS TABLE")
print("=================================================")

display(desc_table)

# =========================================================
# OPTIONAL CSV EXPORT
# =========================================================

save_path = "/content/Descriptive_Statistics_Table.csv"

desc_table.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nTABLE SAVED SUCCESSFULLY")
print("=================================================")
print(save_path)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(
    "/content/HAB_Final_Matched_Dataset_Labeled.csv",
    encoding="utf-8-sig"
)

# =========================================================
# FEATURES
# =========================================================

features = [

    "B11",
    "B2",
    "B2_B3",
    "B3",
    "B3_B2",
    "B3_B4",
    "B4",
    "B5",
    "B5_B4",
    "B8",

    "CI",
    "FAI",
    "NDCI",
    "NDRE",
    "NDVI",
    "NDWI"
]

# =========================================================
# REMOVE NaN
# =========================================================

df_model = df.dropna(
    subset=features + ["HAB_label"]
).copy()

# =========================================================
# INPUTS
# =========================================================

X = df_model[features]

y = df_model["label"]

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

# =========================================================
# BEFORE SMOTE
# =========================================================

print("\nCLASS DISTRIBUTION BEFORE SMOTE")
print("================================")

print(y_train.value_counts().sort_index())

# =========================================================
# SMOTE
# =========================================================

smote = SMOTE(

    sampling_strategy='auto',
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(

    X_train,
    y_train
)

# =========================================================
# AFTER SMOTE
# =========================================================

print("\nCLASS DISTRIBUTION AFTER SMOTE")
print("==============================")

print(
    pd.Series(y_train_smote)
    .value_counts()
    .sort_index()
)

# =========================================================
# SHAPES
# =========================================================

print("\nTRAIN SHAPE AFTER SMOTE")
print("======================")

print(X_train_smote.shape)

print(y_train_smote.shape)

print("\nTEST SHAPE")
print("======================")

print(X_test.shape)

print(y_test.shape)

# =========================================================
# PERCENTAGE
# =========================================================

print("\nTRAIN DISTRIBUTION AFTER SMOTE (%)")
print("==================================")

print(
    (
        pd.Series(y_train_smote)
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

print("\nTEST DISTRIBUTION (%)")
print("====================")

print(
    (
        y_test
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# COUNTS
# =========================================================

train_counts = (
    y_train.value_counts()
    .sort_index()
)

test_counts = (
    y_test.value_counts()
    .sort_index()
)

train_pct = train_counts / train_counts.sum() * 100
test_pct = test_counts / test_counts.sum() * 100

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

blue_colors = [
    "#c6dbef",
    "#6baed6",
    "#08519c"
]

class_labels = [

    "Low bloom\n(<10 μg/L)",

    "Moderate productivity\n(10–75 μg/L)",

    "Elevated Chlorophyll\n(>75 μg/L)"
]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14,7)
)

datasets = [

    ("Training Dataset", train_counts, train_pct),

    ("Testing Dataset", test_counts, test_pct)
]

for ax, (title, counts, pct) in zip(axes, datasets):

    bars = ax.bar(
        range(3),
        counts.values,
        color=blue_colors,
        edgecolor="black"
    )

    ax.set_title(title, fontsize=13)

    ax.set_xticks(range(3))

    ax.set_xticklabels(
        class_labels,
        fontsize=10
    )

    ax.set_ylabel(
        "Number of Samples"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    ymax = counts.max()

    for bar, c, p in zip(
        bars,
        counts.values,
        pct.values
    ):

        ax.text(

            bar.get_x()+bar.get_width()/2,

            bar.get_height()+ymax*0.01,

            f"{c}\n({p:.1f}%)",

            ha="center",

            fontsize=8
        )

plt.suptitle(
    "Training and Testing Dataset Class Distribution",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "Figure_Train_Test_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =========================================================
# COUNTS
# =========================================================

before_counts = (
    y_train.value_counts()
    .sort_index()
)

after_counts = (
    pd.Series(y_train_smote)
    .value_counts()
    .sort_index()
)

before_pct = (
    before_counts /
    before_counts.sum()
    * 100
)

after_pct = (
    after_counts /
    after_counts.sum()
    * 100
)

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

blue_colors = [
    "#c6dbef",
    "#6baed6",
    "#08519c"
]

class_labels = [

    "Low bloom\n(<10 μg/L)",

    "Moderate productivity\n(10-75 μg/L)",

    "Elevated chlorophyll\n(>75 μg/L)"
]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14,7)
)

datasets = [

    ("Before SMOTE", before_counts, before_pct),

    ("After SMOTE", after_counts, after_pct)
]

for ax, (title, counts, pct) in zip(axes, datasets):

    bars = ax.bar(

        range(3),

        counts.values,

        color=blue_colors,

        edgecolor="black"
    )

    ax.set_title(
        title,
        fontsize=13
    )

    ax.set_xticks(
        range(3)
    )

    ax.set_xticklabels(
        class_labels,
        fontsize=10
    )

    ax.set_ylabel(
        "Number of Samples"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    ymax = counts.max()

    for bar, c, p in zip(
        bars,
        counts.values,
        pct.values
    ):

        ax.text(

            bar.get_x()+bar.get_width()/2,

            bar.get_height()+ymax*0.01,

            f"{c}\n({p:.1f}%)",

            ha="center",

            fontsize=8
        )

plt.suptitle(
    "Effect of SMOTE on Training Dataset Class Distribution",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    "Figure_SMOTE_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# CORRELATION COEFFICIENT ANALYSIS
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv("HAB_Final_Matched_Dataset_Labeled.csv")

# =========================================================
# SELECT FEATURES
# =========================================================

features = [
    'B2',
    'B3',
    'B4',
    'B5',
    'B8',
    'B11',
    'B2_B3',
    'B3_B2',
    'B3_B4',
    'B5_B4',
    'CI',
    'FAI',
    'NDCI',
    'NDRE',
    'NDVI',
    'NDWI'
]

X = df[features]

# =========================================================
# PEARSON CORRELATION
# =========================================================

corr_matrix = X.corr(method='pearson')

print("\nPearson Correlation Matrix")
print(corr_matrix.round(3))

# =========================================================
# SAVE TABLE
# =========================================================

corr_matrix.round(4).to_csv(
    "correlation_matrix.csv",
    index=True
)

# =========================================================
# HEATMAP
# =========================================================

fig, ax = plt.subplots(figsize=(12,10))

im = ax.imshow(
    corr_matrix,
    cmap='Blues',
    vmin=-1,
    vmax=1,
    aspect='auto'
)

ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))

ax.set_xticklabels(
    features,
    rotation=90,
    fontsize=10
)

ax.set_yticklabels(
    features,
    fontsize=10
)

# correlation values
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i,j]:.2f}",
            ha='center',
            va='center',
            fontsize=7
        )

cbar = plt.colorbar(im)
cbar.set_label("Pearson Correlation Coefficient (r)")

plt.title(
    "Correlation Matrix of Sentinel-2 Features",
    fontsize=20,
    pad=15
)

plt.tight_layout()

plt.savefig(
    "correlation_heatmap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("\nSaved:")
print("- correlation_matrix.csv")
print("- correlation_heatmap.png")

In [ ]:
# =========================================================
# CORRELATION WITH CHLOROPHYLL-A
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv("HAB_Final_Matched_Dataset_Labeled.csv")

# =========================================================
# FEATURE LIST
# =========================================================

features = [
    'B2',
    'B3',
    'B4',
    'B5',
    'B8',
    'B11',
    'B2_B3',
    'B3_B2',
    'B3_B4',
    'B5_B4',
    'CI',
    'FAI',
    'NDCI',
    'NDRE',
    'NDVI',
    'NDWI'
]

target = 'Chl_a'

# =========================================================
# CORRELATION ANALYSIS
# =========================================================

target_corr = (
    df[features + [target]]
    .corr(method='pearson')[target]
    .drop(target)
    .sort_values(key=np.abs, ascending=False)
)

# =========================================================
# DISPLAY RESULTS
# =========================================================

corr_table = pd.DataFrame({
    'Feature': target_corr.index,
    'Correlation': target_corr.values
})

print("\nCorrelation with Chlorophyll-a")
print(corr_table.round(4))

# =========================================================
# SAVE RESULTS
# =========================================================

corr_table.to_csv(
    "chlorophyll_feature_correlation.csv",
    index=False
)

# =========================================================
# BAR CHART
# =========================================================

plt.figure(figsize=(10, 6))

bars = plt.barh(
    corr_table['Feature'],
    corr_table['Correlation']
)

plt.xlabel("Pearson Correlation Coefficient (r)", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.title(
    "Correlation Between Sentinel-2 Features and Chlorophyll-a",
    fontsize=20,
    pad=15,
)

plt.axvline(
    x=0,
    linestyle='--',
    linewidth=1
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    "chlorophyll_feature_correlation.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# =========================================================
# TOP POSITIVE / NEGATIVE CORRELATIONS
# =========================================================

print("\nTop Positive Correlations")
print(
    target_corr.sort_values(ascending=False)
    .head(5)
    .round(4)
)

print("\nTop Negative Correlations")
print(
    target_corr.sort_values()
    .head(5)
    .round(4)
)

print("\nFiles saved:")
print("- chlorophyll_feature_correlation.csv")
print("- chlorophyll_feature_correlation.png")